## しっぺ返し戦略と裏切り戦略の利得表

In [30]:
class Player:
    def __init__(self, strgFn: callable):
        # プレイヤーが行う戦略更新関数
        self.strgFn = strgFn

        # 初期行動を決める
        # しっぺ返し戦略なら最初は C、裏切り戦略なら D
        if self.strgFn == tit_for_tat:
            self.strg = "C"
        else:
            self.strg = "D"

        # プレイヤーが次にとる行動
        self.nextStrg = None

        # プレイヤーの利得
        self.reset()
        
    def reset(self):
        self.payoff = 0

    def get_payoff(self, p):
        self.payoff += p

    def upd_strg(self) -> None:
        self.strg = self.nextStrg


# しっぺ返し戦略
def tit_for_tat(own: Player, opponent: Player):
    own.nextStrg = opponent.strg

# 裏切り戦略
def all_d(own: Player, opponent: Player):
    own.nextStrg = "D"


In [31]:
# 1対1のしっぺ返し戦略と裏切り戦略のシミュレーション
def simulate(players: list[Player], strgMat: dict, W: float, Tmax: int):
    for t in range(Tmax):
        # print(f"--- {t + 1}回目 ---")

        # 利得の計算
        for i, p in enumerate(players):
            # 1対1なので、自分が0番なら相手は1番、自分が1番なら相手は0番
            opponent = players[1 - i]

            # 自分の行動と相手の行動から利得を取得
            payoff = strgMat[p.strg][opponent.strg] * (W ** t)

            # 利得を加算
            p.get_payoff(payoff)

            # print(f"Player{i + 1}: {p.strg}, payoff +{payoff}")

        # 次の行動を決める
        for i, p in enumerate(players):
            # 相手プレイヤーを取得
            opponent = players[1 - i]

            # 自分の戦略関数に従って nextStrg を決める
            p.strgFn(p, opponent)

        # 全員の行動を同時に更新する
        for p in players:
            p.upd_strg()

strgMat = {
    "C": {"C": 3, "D": 0},
    "D": {"C": 8, "D": 1}
}

tft_1 = Player(tit_for_tat)
tft_2 = Player(tit_for_tat)

all_d_1 = Player(all_d)
all_d_2 = Player(all_d)

W = 0.8
Tmax = 100

# tit_for_tat vs tit_for_tat
tft_1.reset(), tft_2.reset()
simulate([tft_1, tft_2], strgMat, W, Tmax)
print(f"PayOff> TFT1: {tft_1.payoff}, TFT2: {tft_2.payoff}")
# 利得（TFT）: Σ 利得n * 割引率w**t
# 3 / (1 - 0.8) = 15

# tit_for_tat vs all_d
tft_1.reset(), all_d_1.reset()
simulate([tft_1, all_d_1], strgMat, W, Tmax)
print(f"PayOff> TFT1: {tft_1.payoff}, All-D: {all_d_1.payoff}")
# 利得（TFT）: Σ 利得n + 利得t * 割引率w**t （t=1の際に戦略をCからDにするため利得が変わる）
# 0 + 1 * 0.8 / (1 - 0.8)
# 利得（All-D）: Σ 利得n + 利得t * 割引率w**t （t=1の際に相手が戦略をCからDにするため利得が変わる）
# 8 + 1 * 0.8 / (1 - 0.8)

# all_d vs all_d
all_d_1.reset(), all_d_2.reset()
simulate([all_d_1, all_d_2], strgMat, W, Tmax)
print(f"PayOff> All-D1: {all_d_1.payoff}, All-D2: {all_d_2.payoff}")
# 利得（All-D）: Σ 利得n * 割引率w**t
# 1 / (1 - 0.8)

PayOff> TFT1: 14.999999996944451, TFT2: 14.999999996944451
PayOff> TFT1: 3.999999998981482, All-D: 11.999999998981485
PayOff> All-D1: 4.999999998981484, All-D2: 4.999999998981484


## 数式メモ

初回を $t=0$、割引率を $W^t$、繰り返し回数を $T_{\max}$ 

利得行列：

$$
\begin{array}{c|cc}
 & C & D \\
\hline
C & (3, 3) & (0, 8) \\
D & (8, 0) & (1, 1)
\end{array}
$$

---

## 1. TFT vs TFT

しっぺ返し戦略同士の場合、両者とも最初に $C$ を選ぶ。  
その後も相手の $C$ を真似し続けるため、すべての期で $C$ vs $C$ となる。

したがって、各期の利得は $3$ 

$$
V_{\mathrm{TFT},\mathrm{TFT}}
= \sum_{t=0}^{T_{\max}-1} 3W^t
= 3 \sum_{t=0}^{T_{\max}-1} W^t
= 3 \cdot \frac{1 - W^{T_{\max}}}{1 - W}
$$

無限回繰り返しの場合、

$$
V_{\mathrm{TFT},\mathrm{TFT}}
= \frac{3}{1-W}
$$

$W=0.8$ の場合、

$$
V_{\mathrm{TFT},\mathrm{TFT}}
= \frac{3}{1-0.8}
= 15
$$

---

## 2. TFT vs All-D

しっぺ返し戦略は最初に $C$ を選び、裏切り戦略は常に $D$ を選ぶ。

したがって、1期目は

$$
C \text{ vs } D
$$



このとき、TFT 側の利得は $0$、All-D 側の利得は $8$ 

2期目以降、TFT は相手の前回行動 $D$ を真似するため、両者とも $D$ を選ぶ。  
したがって、2期目以降はすべて $D$ vs $D$ となり、各期の利得は $1$ 

### TFT 側の割引利得

$$
V_{\mathrm{TFT},\mathrm{All-D}}
= 0 + \sum_{t=1}^{T_{\max}-1} 1 \cdot W^t
= \sum_{t=1}^{T_{\max}-1} W^t
= \frac{W(1-W^{T_{\max}-1})}{1-W}
$$

無限回繰り返しの場合、

$$
V_{\mathrm{TFT},\mathrm{All-D}}
= \frac{W}{1-W}
$$

$W=0.8$ の場合、

$$
V_{\mathrm{TFT},\mathrm{All-D}}
= \frac{0.8}{1-0.8}
= 4
$$

### All-D 側の割引利得

$$
V_{\mathrm{All-D},\mathrm{TFT}}
= 8 + \sum_{t=1}^{T_{\max}-1} 1 \cdot W^t
= 8 + \sum_{t=1}^{T_{\max}-1} W^t
= 8 + \frac{W(1-W^{T_{\max}-1})}{1-W}
$$

無限回繰り返しの場合、

$$
V_{\mathrm{All-D},\mathrm{TFT}}
= 8 + \frac{W}{1-W}
$$

$W=0.8$ の場合、

$$
V_{\mathrm{All-D},\mathrm{TFT}}
= 8 + \frac{0.8}{1-0.8}
= 12
$$

---

## 3. All-D vs All-D

裏切り戦略同士の場合、両者とも常に $D$ を選ぶ。

したがって、すべての期で $D$ vs $D$ となり、各期の利得は $1$ 

$$
V_{\mathrm{All-D},\mathrm{All-D}}
= \sum_{t=0}^{T_{\max}-1} 1 \cdot W^t
= \sum_{t=0}^{T_{\max}-1} W^t
= \frac{1 - W^{T_{\max}}}{1-W}
$$

無限回繰り返しの場合、

$$
V_{\mathrm{All-D},\mathrm{All-D}}
= \frac{1}{1-W}
$$

$W=0.8$ の場合、

$$
V_{\mathrm{All-D},\mathrm{All-D}}
= \frac{1}{1-0.8}
= 5
$$

---

## 4. 利得表

$$
\begin{array}{c|cc}
 & \mathrm{TFT} & \mathrm{All-D} \\
\hline
\mathrm{TFT}
& (15, 15)
& (4, 12) \\
\mathrm{All-D}
& (12, 4)
& (5, 5)
\end{array}
$$


## 得点依存増殖率モデルと得点依存生存率モデル

- 得点依存増殖率モデル
ランダムなプレイヤーが死亡 & 新たに生まれるプレイヤーの戦略は利得に依存するモデル

- 得点依存生存率モデル
利得に依存してプレイヤーが死亡 & 新たに生まれるプレイヤーの戦略はランダムなモデル

In [32]:
import math
import random

# 繰り返し囚人のジレンマゲーム
class IPD:
    def __init__(self, strgFn_list: list[callable], payMat: dict, N_grid: int, w: float):
        self.strgFn_list = strgFn_list # 戦略のリスト
        self.payMat = payMat # 利得行列
        self.player_list = [Player(random.choice(self.strgFn_list)) for _ in range(N_grid)] # プレイヤーのリスト
        self.w = w # 反復確率
        self.reset()

    def reset(self):
        for player in self.player_list:
            player.reset()

    def get_payoff(self, own: Player, neighbor: Player):
        # TFT vs TFT or ALL-D vs ALL-D の場合
        if own.strgFn == neighbor.strgFn:
            payoff = self.payMat[own.strg][neighbor.strg] / (1 - self.w)
        # TFT vs ALL-D の場合
        else:
            payoff = (self.payMat[own.strg][neighbor.strg]) + (self.payMat["D"]["D"] * self.w / (1 - self.w))
        own.get_payoff(payoff)

    def play_game(self):
        for i, player in enumerate(self.player_list):
            # 隣接するプレイヤーを取得
            neighbor_1 = self.player_list[(i - 1) % len(self.player_list)] # 左隣
            neighbor_2 = self.player_list[(i + 1) % len(self.player_list)] # 右隣

            # 自分と隣人の行動から利得を取得
            self.get_payoff(player, neighbor_1)
            self.get_payoff(player, neighbor_2)

class GRDS(IPD):
    """
    Growth Rate Dependent Score
    """
    def __init__(self, strgFn_list: list[callable], payMat: dict, N_grid: int, w: float):
        super().__init__(strgFn_list, payMat, N_grid, w)

    def dropout(self):
        # ランダムなプレイヤーを選択
        dropout_player = random.choice(self.player_list)
        # 選択されたプレイヤーを None に置き換える
        self.player_list[self.player_list.index(dropout_player)] = None

    def generate(self):
        for i, player in enumerate(self.player_list):
            # None のプレイヤーを新しいプレイヤーで置き換える
            if player is None:
                # 新しいプレイヤーのstrgFnは、隣接するプレイヤーの利得に基づいて確率的に選択される
                neighbor_1 = self.player_list[(i - 1) % len(self.player_list)] # 左隣
                neighbor_2 = self.player_list[(i + 1) % len(self.player_list)] # 右隣

                # 新しいプレイヤーが1の戦略を真似る確率を計算（ロジスティック関数）
                payoff_diff = neighbor_1.payoff - neighbor_2.payoff
                rate_1 = 1 / (1 + math.exp(-payoff_diff))
                # 新しいプレイヤーの戦略を確率的に選択
                if random.random() < rate_1:
                    strgFn= neighbor_1.strgFn
                else:
                    strgFn = neighbor_2.strgFn
                self.player_list[i] = Player(strgFn)

class SRDS(IPD):
    """
    Survival Rate Dependent Score
    """
    def __init__(self, strgFn_list: list[callable], payMat: dict, N_grid: int, w: float):
        super().__init__(strgFn_list, payMat, N_grid, w)

    def dropout(self):
        # ドロップアウトするプレイヤーは利得に依存する　※利得が低いプレイヤーほどドロップアウトする確率が高い
        # 各プレイヤーがドロップアウトする確率を計算（ソフトマックス関数）
        active_players = [(i, player) for i, player in enumerate(self.player_list) if player is not None]
        if not active_players:
            return

        # 利得が低いほど重みが大きくなるように、-payoff に softmax を適用する
        scores = [-player.payoff for _, player in active_players]
        max_score = max(scores) # オーバーフロー対策
        weights = [math.exp(score - max_score) for score in scores]
        total_weight = sum(weights)
        dropout_probs = [weight / total_weight for weight in weights]

        dropout_index = random.choices(
            [i for i, _ in active_players],
            weights=dropout_probs,
            k=1
        )[0]

        # 選択されたプレイヤーを None に置き換える
        self.player_list[dropout_index] = None

    def generate(self):
        for i, player in enumerate(self.player_list):
            # None のプレイヤーを新しいプレイヤーで置き換える
            if player is None:
                # 新しいプレイヤーのstrgFnは、ランダムに選択される
                self.player_list[i] = Player(random.choice(self.strgFn_list))

In [ ]:
def calc_tft_frequency(player_list: list[Player]) -> float:
    """Tit for Tat (TFT) の割合を計算する"""
    nTFT = 0 # TFT人数
    for player in player_list:
        if player.strgFn == tit_for_tat:
            nTFT += 1
    
    return nTFT / (len(player_list))

def simulate(game: GRDS | SRDS, Tmax: int=1000):
    # TFT の割合リスト
    tft_ratio_list = []

    for i in range(Tmax):

        tft_ratio_list.append(calc_tft_frequency(game.player_list))

        # 利得の初期化
        game.reset()
        # ゲームをして利得を獲得
        game.play_game()
        # ドロップアウト
        game.dropout()
        # ドロップアウトしたところに生成
        game.generate()

    return tft_ratio_list


    


N = 100 # 格子点
w = 0.8 # 反復確率
payMat = {
    "C": {"C": 3, "D": 0},
    "D": {"C": 8, "D": 1}
}
Game = GRDS([tit_for_tat, all_d], payMat, N, w)

tft_ratio_list = simulate(Game)

# print(tft_ratio_list)

[0.5, 0.51, 0.51, 0.52, 0.52, 0.52, 0.51, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.51, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.51, 0.51, 0.51, 0.51, 0.51, 0.51, 0.52, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.53, 0.54, 0.55, 0.55, 0.55, 0.56, 0.56, 0.55, 0.55, 0.55, 0.56, 0.56, 0.56, 0.56, 0.56, 0.56, 0.56, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.54, 0.54, 0.54, 0.54, 0.54, 0.54, 0.54, 0.55, 0.55, 0.55, 0.55, 0.56, 0.56, 0.56, 0.56, 0.56, 0.56, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.56, 0.56, 0.56, 0.56, 0.56, 0.56, 0.57, 0.57, 0.57, 0.57, 0.57, 0.58, 0.58, 0.58, 0.58, 0.59, 0.59, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.59, 0.59, 0.59, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.61, 0.61, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.61, 0.61, 0.61, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.62, 0.63, 0.63, 0.63, 0.63, 0.63, 0.63, 0.63, 0.64, 0.65, 0.65, 0.65, 0.65, 0.66, 0.66, 